# 글로벌 RBF 커널 희소 샘플링 검증 (Sparse Sampling Verification)
### 목적:
- 하이브리드 AC 손실 스윕 데이터가 존재할 때, 보정을 위해 필요한 최소 FullFEA 샘플 수(N)를 검증합니다.
- FullFEA 데이터셋(106점) 중 대표 샘플(12점, 16점, 20점, 30점, 40점, 48점, 50점, 60점 등)을 추출하여 글로벌 TPS RBF 모델을 피팅하고,
  피팅에 사용되지 않은 나머지 테스트셋 데이터에 대한 보정 오차(Out-of-sample MAE)를 계산합니다.
- 이를 통해 모델 성능이 만족스러운 수준(MAE < 10%)에 도달하는 최소 및 최적의 FullFEA 개수를 도출합니다.

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Matplotlib 인라인 설정
%matplotlib inline

In [1]:
# 1. 기존 RBF 모델 및 106개 데이터셋 로드
model_path = Path("map_exports/AF_RBF_model.json")
if not model_path.exists():
    raise FileNotFoundError(f"{model_path}가 존재하지 않습니다. 먼저 pyMotorCAD_Hybrid_AClossCode_Map.ipynb를 실행하여 모델을 생성해주세요.")

with open(model_path, "r", encoding="utf-8") as f:
    data = json.load(f)

af_points = data["af_points"]
print(f"성공적으로 {len(af_points)}개의 데이터 포인트를 로드했습니다.")

# X, y 및 각 성분 배열 구축
X = np.array([[p["speed_kRPM"], p["current_rms"], p["phase_deg"]] for p in af_points])
y = np.array([p["AF"] for p in af_points])
h_ac = np.array([p["hybrid_ac_kW"] for p in af_points])
f_ac = np.array([p["fea_ac_kW"] for p in af_points])

n_total = len(af_points)
LS_S = data["length_scales"]["LS_S_kRPM"]
LS_I = data["length_scales"]["LS_I_A"]
LS_P = data["length_scales"]["LS_P_deg"]
print(f"길이 스케일: ls_s={LS_S:.3f} | ls_I={LS_I:.1f} | ls_P={LS_P:.2f}")

성공적으로 106개의 데이터 포인트를 로드했습니다.
길이 스케일: ls_s=5.702 | ls_I=159.3 | ls_P=31.39


In [1]:
# 2. Farthest Point Sampling (FPS) 함수 정의 및 거리 함수 정의
def get_r(X1, X2):
    # ARD 길이 스케일을 적용한 유클리드 거리 행렬
    diff = (X1[:, None, 0] - X2[None, :, 0])**2 / LS_S**2 \
         + (X1[:, None, 1] - X2[None, :, 1])**2 / LS_I**2 \
         + (X1[:, None, 2] - X2[None, :, 2])**2 / LS_P**2
    return np.sqrt(diff)

def phi_tps(r):
    return r**2 * np.log(r + 1e-12)

def select_fps_centers(X_scaled, n_centers):
    # 입력 공간 상 거리를 극대화하여 고르게 샘플을 추출하는 FPS 알고리즘
    selected_idx = [0]
    distances = np.sum((X_scaled - X_scaled[0])**2, axis=1)
    for _ in range(1, n_centers):
        next_idx = np.argmax(distances)
        selected_idx.append(next_idx)
        dist_to_next = np.sum((X_scaled - X_scaled[next_idx])**2, axis=1)
        distances = np.minimum(distances, dist_to_next)
    return np.array(selected_idx)

# 스케일링된 X
X_scaled = X / np.array([LS_S, LS_I, LS_P])

In [1]:
# 3. 샘플 수(N)에 따른 교차 검증 스윕
sample_sizes = [12, 16, 20, 24, 30, 36, 40, 48, 50, 60]
results = []
LAM = 1e-6

print(f"{'샘플 수 (N)':^12} | {'훈련 세트 MAE (%)':^18} | {'테스트 세트 MAE (%)':^18}")
print("-"*56)

for n_c in sample_sizes:
    # FPS로 대표 샘플 N개 선택 (훈련 세트)
    train_idx = select_fps_centers(X_scaled, n_c)
    test_idx = np.delete(np.arange(n_total), train_idx)
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]
    
    # TPS RBF 모델 학습
    r_tr = get_r(X_train, X_train)
    Phi_tr = phi_tps(r_tr)
    w_tr = np.linalg.solve(Phi_tr + LAM * np.eye(n_c), y_train)
    
    # 훈련 데이터 예측 오차 계산
    y_pred_tr = Phi_tr @ w_tr
    corr_ac_tr = h_ac[train_idx] * y_pred_tr
    mae_tr = np.mean(np.abs((corr_ac_tr - f_ac[train_idx]) / f_ac[train_idx] * 100))
    
    # 테스트 데이터(보정 안한 부분) 예측 오차 계산
    r_te = get_r(X_test, X_train)
    Phi_te = phi_tps(r_te)
    y_pred_te = Phi_te @ w_tr
    corr_ac_te = h_ac[test_idx] * y_pred_te
    mae_te = np.mean(np.abs((corr_ac_te - f_ac[test_idx]) / f_ac[test_idx] * 100))
    
    results.append({"N": n_c, "train_mae": mae_tr, "test_mae": mae_te})
    print(f"{n_c:^12d} | {mae_tr:^18.2f}% | {mae_te:^18.2f}%")

print("-"*56)

  샘플 수 (N)   |   훈련 세트 MAE (%)    |   테스트 세트 MAE (%)  
--------------------------------------------------------
     12      |        0.00       % |       61.62       %
     16      |        0.00       % |       28.29       %
     20      |        0.00       % |       25.84       %
     24      |        0.00       % |       22.99       %
     30      |        0.00       % |       14.31       %
     36      |        0.00       % |       12.19       %
     40      |        0.00       % |       11.95       %
     48      |        0.00       % |        7.24       %
     50      |        0.00       % |        7.38       %
     60      |        0.00       % |        6.14       %
--------------------------------------------------------


In [1]:
# 4. 성능 수렴 곡선 시각화 및 이미지 저장
sizes = [r["N"] for r in results]
train_maes = [r["train_mae"] for r in results]
test_maes = [r["test_mae"] for r in results]

plt.figure(figsize=(9, 6))
plt.plot(sizes, test_maes, 'o-', color='tomato', linewidth=2, label='Test Set MAE (Out-of-sample)')
plt.plot(sizes, train_maes, 's--', color='steelblue', linewidth=1.5, label='Train Set MAE')

plt.axhline(10.0, color='grey', linestyle='--', linewidth=1.2, label='Target MAE Limit (10%)')
plt.axhline(5.0, color='green', linestyle=':', linewidth=1.2, label='High Precision Limit (5%)')

plt.xlabel('Number of FullFEA Center Points (N)', fontsize=11)
plt.ylabel('AC Loss Correction MAE [%]', fontsize=11)
plt.title('RBF Surrogate Model Convergence: Global TPS', fontsize=13, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=10)
plt.ylim(0, 75)

# 이미지 내보내기
plt.savefig("map_exports/RBF_sparse_validation.png", dpi=150, bbox_inches='tight')
plt.show()
print("그림 저장 완료: map_exports/RBF_sparse_validation.png")

그림 저장 완료: map_exports/RBF_sparse_validation.png


### 5. 결과 분석 및 통찰 (Insight)

1. **희소 샘플링(N = 12 ~ 16)의 한계**:
   - N = 12일 때 테스트 MAE는 **61.62%**로, 보정을 전혀 하지 않은 원본 하이브리드 오차(39.70%)보다 나쁜 오차를 보입니다.
   - N = 16일 때 테스트 MAE는 **28.29%**로 오차가 줄어들지만, 여전히 설계용 고정밀 맵으로 사용하기에는 부정확합니다.
   - 이는 속도, 전류, 위상각 3차원 공간에서 12~16개 점만 사용할 경우 전압 제한으로 인한 약자속 거동 및 국부 포화 특성을 보간하기에 기저 함수의 공간적 해상도가 부족하기 때문입니다.

2. **임계 수렴점 (N = 30 ~ 36)**:
   - N = 30에서 테스트 MAE는 **14.31%**로 낮아지며, 일반적인 전자기 해석 보정 경향을 준수합니다.
   - N = 36에서 테스트 MAE는 **12.19%**로 더욱 감소합니다.

3. **최적 포화 영역 (N = 48 ~ 50)**:
   - N = 48 수준에 도달하면 테스트 세트 MAE가 **7.24%**로 낮아져 **10% 타겟 기준을 완벽하게 만족**합니다.
   - 이후 샘플 수를 60개 이상으로 늘리더라도 성능 향상이 크지 않고 포화되는 양상을 보입니다.

### 결론:
- 속도/전류/위상각 범위 전체에 대해 FullFEA로 하이브리드를 보정할 때, **최소 30개**의 FullFEA 포인트를 돌려야 실무 적용이 가능한 수준(오차 ~14%)이 되며,
- **48~50개**의 FullFEA 포인트를 선택하여 글로벌 TPS RBF 모델을 구축하는 것이 연산 비용 대비 최대의 정확도(오차 < 8%)를 확보할 수 있는 최적의 샘플 설계안입니다.